# HomeBot-Sim: headless E2 eval (MuJoCo, EGL) for ACT / N1.7

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
DATA = "/kaggle/input/homebot-can"          # Kaggle Dataset made from data/homebot_can
!nvidia-smi

In [ ]:
%env MUJOCO_GL=egl
!pip install -q mujoco==3.11.0 scipy pandas pyarrow imageio imageio-ffmpeg opencv-python-headless
!git clone --depth 1 https://github.com/<your-user>/homebot-sim /kaggle/working/homebot-sim
%cd /kaggle/working/homebot-sim
!bash scripts/setup_assets.sh        # Menagerie G1, Inspire hand meshes, YCB soup can (~230 MB)

N1.7: start the Isaac-GR00T inference server on GPU 0 in the background, then evaluate.

In [ ]:
import subprocess
srv = subprocess.Popen("cd /kaggle/working/Isaac-GR00T && python scripts/inference_service.py --server "
                       "--model-path /kaggle/working/n17_homebot --embodiment-tag new_embodiment "
                       "--data-config /kaggle/working/homebot-sim/configs/gr00t_g1_sonic_inspire.py:make_config",
                       shell=True)
!python scripts/eval_manip.py --n 50 --policy gr00t --host localhost
!python scripts/eval_manip.py --n 50 --policy torch:/kaggle/working/act_homebot/act_traced.pt
!python scripts/plots.py